# MIVI-V2 Agentic Finetune — MiniCPM5-1B + LFM2.5-350MTrains both candidate bases on `datasets/mivi_agentic_sft.jsonl` (182 rows oftool-calling / planner / summarization / identity / chat behaviors).**Setup:** Runtime -> Change runtime type -> **T4 GPU**. Then Run All.Each run takes ~10-15 min. Outputs land in `outputs/` — download the `.gguf`files at the last cell and drop them into MIVI's `models/`.

In [ ]:
%%capture!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"!pip install --no-deps trl peft accelerate bitsandbytes

## Dataset — upload `datasets/mivi_agentic_sft.jsonl` when prompted

In [ ]:
from google.colab import filesimport pathlibup = files.upload()  # pick datasets/mivi_agentic_sft.jsonlpathlib.Path('datasets').mkdir(exist_ok=True)name = list(up.keys())[0]import shutil, osshutil.move(name, f'datasets/{name}')DATASET = f'datasets/{name}'print('dataset:', DATASET, '| rows:', sum(1 for _ in open(DATASET)))

## Shared trainer (from `scripts/train_mivi_unsloth.py`)

In [ ]:
import sys, jsonsys.path.insert(0, 'scripts')# If you cloned the repo this file already exists; otherwise it is inlined below.USE_REPO_SCRIPT = pathlib.Path('scripts/train_mivi_unsloth.py').exists()def run_training(base_model, output_dir, max_steps=60):    '''Thin wrapper: calls the repo script when present, else inline Unsloth flow.'''    if USE_REPO_SCRIPT:        import subprocess        subprocess.run([sys.executable, 'scripts/train_mivi_unsloth.py',                        '--base_model', base_model,                        '--dataset_path', DATASET,                        '--output_dir', output_dir,                        '--max_steps', str(max_steps),                        '--export_gguf', '1'], check=True)        return    # Inline fallback (same logic as scripts/train_mivi_unsloth.py)    from unsloth import FastLanguageModel    import torch    from datasets import Dataset    from trl import SFTTrainer    from transformers import TrainingArguments    model, tokenizer = FastLanguageModel.from_pretrained(        model_name=base_model, max_seq_length=4096, dtype=None, load_in_4bit=True)    model = FastLanguageModel.get_peft_model(        model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj",                                     "gate_proj","up_proj","down_proj"],        lora_alpha=32, lora_dropout=0, bias="none",        use_gradient_checkpointing="unsloth", random_state=3407)    raw = [json.loads(l) for l in open(DATASET) if l.strip()]    texts = [{"text": tokenizer.apply_chat_template(        it["messages"], tokenize=False, add_generation_prompt=False)} for it in raw]    trainer = SFTTrainer(model=model, tokenizer=tokenizer,        train_dataset=Dataset.from_list(texts), dataset_text_field="text",        max_seq_length=4096, packing=False,        args=TrainingArguments(per_device_train_batch_size=4,            gradient_accumulation_steps=4, warmup_steps=10, max_steps=max_steps,            learning_rate=2e-4, fp16=not torch.cuda.is_bf16_supported(),            bf16=torch.cuda.is_bf16_supported(), logging_steps=10,            optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",            seed=3407, output_dir=output_dir, report_to="none"))    trainer.train()    model.save_pretrained_merged(output_dir, tokenizer,                                 save_method="merged_16bit")    print('done:', output_dir)

## Run A — MiniCPM5-1B (primary candidate, apache-2.0)

In [ ]:
run_training('openbmb/MiniCPM5-1B', 'outputs/mivi-minicpm5-agent', max_steps=60)

## Run B — LFM2.5-350M (ultra-light candidate, 438 MB inference)

In [ ]:
run_training('LiquidAI/LFM2.5-350M', 'outputs/mivi-lfm350-agent', max_steps=60)

## Export — download both GGUFs

In [ ]:
import glob, osfrom google.colab import filesggufs = glob.glob('outputs/**/*.gguf', recursive=True)print('found:', ggufs)for g in ggufs:    files.download(g)  # drop these into mivi-v2/models/

## Local evaluation (back on your machine)```bashcp <downloaded>.gguf models/MIVI_RUNTIME_MODE=worker-eco \MIVI_REASONER_MODEL=models/<minicpm5-agent>.gguf \MIVI_CODER_MODEL=models/<minicpm5-agent>.gguf \just agent-eval# then same for the lfm350 gguf; winner = higher score (target >= 8/11)```Baseline to beat: Qwen3-1.7B default = **7/11**. Pre-finetune: MiniCPM5-1B = 6/11, LFM2.5-350M = 6/11.